In [ ]:
# Generate Weather GIF with Enhanced Annotations and Styling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import imageio.v2 as imageio
from shapely.geometry import Polygon, box
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.cm as cm

# === Constants ===
dtw_lat, dtw_lon = 42.2125, -83.3534
radius_deg = 3
grid_step = 0.7

star_fixes = {
    "BONZZ": (-82.7972, 41.7483), "CRAKN": (-82.9405, 41.6730), "CUUGR": (-83.0975, 42.3643),
    "FERRL": (-82.6093, 42.4165), "GRAYT": (-83.6020, 42.9150), "HANBL": (-84.1773, 41.7375),
    "HAYLL": (-84.2975, 41.9662), "HTROD": (-83.3442, 42.0278), "KKISS": (-83.7620, 42.5443),
    "KLYNK": (-82.9888, 41.8793), "LAYKS": (-83.5498, 42.8532), "LECTR": (-84.0217, 41.9183),
    "RKCTY": (-83.9603, 42.6869), "VCTRZ": (-84.0670, 41.9878)
}
ordered_fix_names = ["CRAKN", "BONZZ", "FERRL", "LAYKS", "GRAYT", "RKCTY", "HAYLL", "HANBL"]
tracon_polygon = Polygon([star_fixes[name] for name in ordered_fix_names])

theta = np.linspace(0, 2 * np.pi, 400)
circle_coords = [(dtw_lon + radius_deg * np.cos(t), dtw_lat + radius_deg * np.sin(t)) for t in theta]
circle_polygon = Polygon(circle_coords)
minx, miny, maxx, maxy = circle_polygon.bounds

# === Grid Cells ===
clipped_cells = []
for lon in np.arange(minx, maxx, grid_step):
    for lat in np.arange(miny, maxy, grid_step):
        cell = box(lon, lat, lon + grid_step, lat + grid_step)
        if circle_polygon.intersects(cell) and not tracon_polygon.contains(cell):
            clipped = cell.intersection(circle_polygon)
            if not clipped.is_empty:
                clipped_cells.append(clipped)

# === Load weather CSV ===
csv_path = "/Users/hoo/Desktop/UMICH/FAAFAST/radar_imgs_20231222_0000_0015/weather_grid_20231222_0000_0015_labeled.csv"
weather_df = pd.read_csv(csv_path)
weather_df['time'] = pd.to_datetime(weather_df['time'])
time_points = sorted(weather_df['time'].unique())

# === Setup output ===
output_folder = "frames_weather_pretty"
os.makedirs(output_folder, exist_ok=True)
filenames = []

# === Create Frames ===
for idx, t in enumerate(time_points):
    frame = weather_df[weather_df['time'] == t].sort_values("grid_id")
    dbz_vals = frame['mean_dbz'].values
    vmax = max(60, np.nanmax(dbz_vals))

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(minx - 0.1, maxx + 0.1)
    ax.set_ylim(miny - 0.1, maxy + 0.1)
    ax.set_aspect('equal')
    ax.set_title("Grid Weather Reflectivity", fontsize=16, fontweight='bold')

    # Fill grid
    norm = Normalize(vmin=0, vmax=vmax)
    cmap = cm.get_cmap("Blues")
    colors = cmap(norm(dbz_vals))

    for i, poly in enumerate(clipped_cells):
        if isinstance(poly, Polygon):
            x, y = poly.exterior.xy
            ax.fill(x, y, facecolor=colors[i], edgecolor='gray', linewidth=0.5, zorder=2)

    # Pre-TRACON Circle
    circle_x, circle_y = zip(*circle_coords)
    ax.plot(circle_x, circle_y, linestyle='--', color='black', linewidth=1, label='Pre-TRACON Boundary', zorder=1)

    # TRACON polygon border only
    tracon_patch = MplPolygon(tracon_polygon.exterior.coords, closed=True,
                              edgecolor='blue', facecolor='none', linewidth=2, label='TRACON', zorder=3)
    ax.add_patch(tracon_patch)

    # STAR fixes and labels
    for name, (lon, lat) in star_fixes.items():
        ax.plot(lon, lat, 'ko', markersize=4, zorder=4)
        ax.text(lon + 0.015, lat + 0.01, name, fontsize=7, color='black', zorder=5)

    # KDTW
    ax.plot(dtw_lon, dtw_lat, 'r*', markersize=12, zorder=5)
    ax.text(dtw_lon + 0.02, dtw_lat, 'KDTW', fontsize=10, fontweight='bold', zorder=5)

    # Time annotation
    ax.text(0.5, -0.1, f"Time: {pd.Timestamp(t).strftime('%Y-%m-%d %H:%M UTC')}", fontsize=12,
            transform=ax.transAxes, ha='center', va='center', bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="gray"))

    # Colorbar with explanation
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.85)
    cbar.set_label("Mean Reflectivity (dBZ)
(>15 dBZ = Moderate Rain)", fontsize=10)

    fig.tight_layout()
    fname = f"{output_folder}/frame_{idx:03d}.png"
    plt.savefig(fname)
    plt.close()
    filenames.append(fname)

# === Save GIF ===
gif_path = "Weather_Grid_Evolution_Pretty.gif"
imageio.mimsave(gif_path, [imageio.imread(f) for f in filenames], fps=1)
gif_path